In [7]:
import os
from pathlib import Path
import re
import pandas as pd
import jsonlines
import ast

PATH = r"C:\Users\emste\Documents\cloned_Gits\bachelorthesis_multilingual_steering\results\or_bench_L11_S1.0\meta-llama__Llama-3.1-8B-Instruct\samples_or_bench_2025-08-07T23-07-12.032751.jsonl"

def extract_lang(x):
    """Extract language id for all samples from output"""
    try:
        if isinstance(x, str):
            x = ast.literal_eval(x)
        return x.get("id", "")[-2:]  # last two characters of 'id' are ISO codes for that language
    except Exception as e:
        print(f"Error processing: {x}, Error: {e}") 
        return None  
    
def read_in_jsonl_to_df(filepath):
    """Read in jsonl object and turn to dataframe.
    - Clean LLM responses
    - Add col for combined prompt and LLM answer as input for LLM judge"""

    # jsonl to df
    data = []
    with jsonlines.open(filepath) as reader:
        for obj in reader:
            data.append(obj)

    df = pd.DataFrame(data)

    # clean and add combined prompt + answer prompt as new col
    df["filtered_resps"] = (
        df["filtered_resps"]
        .astype(str)
        .str.replace("[", "", regex=False)
        .str.replace("]", "", regex=False)
    )
    df["prompt"] = df["doc"].apply(lambda x: x["prompt"])
    df["prompt_and_answer"] = (
        "Query: " + df["prompt"] + " Response: " + df["filtered_resps"]
    )

    return df

def save_pretty_output_df(full_df, task_name, out_path):
    """Save a df with answers to the specified path. It also makes the output df more readable for analysis"""
    print(f"All columns before processing: {full_df.columns}")

    # In the way lm-eval returns the samples the [doc] column is a json with different values we unpack here.
    doc_df = pd.json_normalize(full_df['doc'])
    full_df = pd.concat([full_df.drop('doc', axis=1), doc_df], axis=1)


    print(f"All columns AFTER processing (should be p-id, lang, prompt, answer_og, answer_translated)\n: {full_df.columns}")
    print(f"Head: {full_df["category"].head(10)}")
    full_out_path = f"{out_path}{task_name}_translated.csv"
    # full_df.to_csv(full_out_path)
    print(f"Done! Finished dataframe has been written to: {full_out_path}")

df = read_in_jsonl_to_df(PATH)
save_pretty_output_df(df, "or_bench_L11_S1.0", "C:/Users/emste/Documents/cloned_Gits/bachelorthesis_multilingual_steering/scripts/")


All columns before processing: Index(['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps',
       'filter', 'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'bypass',
       'prompt', 'prompt_and_answer'],
      dtype='object')
All columns AFTER processing (should be p-id, lang, prompt, answer_og, answer_translated)
: Index(['doc_id', 'target', 'arguments', 'resps', 'filtered_resps', 'filter',
       'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'bypass', 'prompt',
       'prompt_and_answer', 'id', 'lang', 'text', 'prompt', 'category'],
      dtype='object')
Head: 0    None
1    None
2    None
3    None
4    None
5    None
6    None
7    None
8    None
9    None
Name: category, dtype: object
Done! Finished dataframe has been written to: C:/Users/emste/Documents/cloned_Gits/bachelorthesis_multilingual_steering/scripts/or_bench_L11_S1.0_translated.csv
